# Segmentación background/foreground

Este notebook:

1. Carga una imagen en color RGB.
2. Aplica un filtro Gaussiano pequeño sobre la imagen RGB.
3. Convierte la imagen RGB filtrada al espacio de color HSV.
4. Clasifica **background** y **foreground** con:
   - `KNeighborsClassifier` de **scikit-learn**
   - `cv2.ml.KNearest` de **OpenCV**

La estrategia de entrenamiento usa semillas automáticas simples:
- **background**: píxeles de las esquinas
- **foreground**: píxeles de una región central

Puede ajustar el tamaño de las semillas según su imagen.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['image.cmap'] = 'gray'

## Parámetros

Cambie la ruta de la imagen por la suya.


In [ ]:
# Ruta de entrada
IMAGE_PATH = 'leaf_1.JPG'  # Cambie por su archivo

# Parámetros del filtro Gaussiano
GAUSSIAN_KERNEL = (5, 5)
GAUSSIAN_SIGMA = 2

# Parámetros KNN
K_NEIGHBORS = 5

# Fracción usada para construir semillas automáticas
# Esquinas -> background
CORNER_FRACTION = 0.1

# Región central -> foreground
CENTER_FRACTION = 0.15

## Cargar imagen RGB y mostrarla

In [ ]:
img_bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
if img_bgr is None:
    raise FileNotFoundError(f'No se pudo cargar la imagen: {IMAGE_PATH}')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
h, w, _ = img_rgb.shape

print(f'Tamaño de imagen: {w} x {h}')

plt.imshow(img_rgb)
plt.title('Imagen RGB')
plt.axis('off')
plt.show()

## Filtrado de la imagen RGB y cambio de espacio de color

En esta versión, el filtro Gaussiano se aplica únicamente sobre la imagen RGB.
Recuerde que la imagen HSV **no** se filtra directamente.

In [ ]:
img_rgb_blur = cv2.GaussianBlur(img_rgb, GAUSSIAN_KERNEL, GAUSSIAN_SIGMA)
img_hsv = cv2.cvtColor(img_rgb_blur, cv2.COLOR_RGB2HSV)

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(img_rgb)
ax[0].set_title('RGB original')
ax[0].axis('off')

ax[1].imshow(img_rgb_blur)
ax[1].set_title('RGB filtrada con Gaussiano')
ax[1].axis('off')

# Para visualizar HSV en matplotlib normalmente se convierte otra vez a RGB
hsv_to_rgb_vis = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2RGB)
ax[2].imshow(hsv_to_rgb_vis)
ax[2].set_title('HSV obtenida desde RGB filtrada')
ax[2].axis('off')
plt.show()

## Crear semillas automáticas de entrenamiento

Etiqueta usada:
- `0` = background
- `1` = foreground

In [ ]:
def build_seed_masks(height, width, corner_fraction=0.15, center_fraction=0.30):
    bg_mask = np.zeros((height, width), dtype=np.uint8)
    fg_mask = np.zeros((height, width), dtype=np.uint8)

    ch = max(1, int(height * corner_fraction))
    cw = max(1, int(width * corner_fraction))

    # Cuatro esquinas como background
    bg_mask[:ch, :cw] = 1
    bg_mask[:ch, width-cw:] = 1
    bg_mask[height-ch:, :cw] = 1
    bg_mask[height-ch:, width-cw:] = 1

    # Región central como foreground
    fh = max(1, int(height * center_fraction))
    fw = max(1, int(width * center_fraction))
    y0 = (height - fh) // 2
    x0 = (width - fw) // 2
    fg_mask[y0:y0+fh, x0:x0+fw] = 1

    # Evitar solapamiento
    fg_mask[bg_mask == 1] = 0

    return bg_mask, fg_mask

Generación de semillas

In [ ]:
bg_seed_mask, fg_seed_mask = build_seed_masks(
    h, w,
    corner_fraction=CORNER_FRACTION,
    center_fraction=CENTER_FRACTION,
)

seed_vis = img_rgb.copy()
seed_vis[bg_seed_mask == 1] = [255, 0, 0]   # rojo = background
seed_vis[fg_seed_mask == 1] = [0, 255, 0]   # verde = foreground

plt.imshow(seed_vis)
plt.title('Semillas automáticas: rojo=background, verde=foreground')
plt.axis('off')
plt.show()

print('Píxeles background:', int(bg_seed_mask.sum()))
print('Píxeles foreground:', int(fg_seed_mask.sum()))

## Preparar características y etiquetas

Este fragmento prepara los datos de entrenamiento para la segmentación supervisada de la imagen. Primero, reorganiza todos los píxeles de la imagen en espacio HSV como una lista de características, donde cada píxel queda representado por sus tres componentes de color. Luego, identifica cuáles píxeles fueron marcados previamente como fondo y cuáles como objeto de interés mediante las máscaras semilla, y con esa información construye el conjunto de entrenamiento X_train junto con sus etiquetas y_train, asignando 0 al fondo y 1 al primer plano.

In [ ]:
features = img_hsv.reshape(-1, 3).astype(np.float32)

bg_idx = np.where(bg_seed_mask.reshape(-1) == 1)[0]
fg_idx = np.where(fg_seed_mask.reshape(-1) == 1)[0]

X_train = np.vstack([features[bg_idx], features[fg_idx]])
y_train = np.hstack([
    np.zeros(len(bg_idx), dtype=np.uint8),
    np.ones(len(fg_idx), dtype=np.uint8)
])

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

## Clasificación con scikit-learn

Entrena y aplica el clasificador KNN para realizar la segmentación de la imagen. Primero, se crea el modelo indicando cuántos vecinos cercanos se tendrán en cuenta, y luego se ajusta usando los píxeles etiquetados previamente como fondo y primer plano. Después, el modelo clasifica todos los píxeles de la imagen según su similitud con esos ejemplos de entrenamiento, generando una predicción para cada uno. Finalmente, esas predicciones se reorganizan con la forma original de la imagen para construir la máscara segmentada y se visualiza el resultado, mostrando qué regiones fueron identificadas como pertenecientes al objeto de interés y cuáles al fondo.

In [ ]:
knn_sklearn = KNeighborsClassifier(n_neighbors=K_NEIGHBORS)
knn_sklearn.fit(X_train, y_train)

pred_sklearn = knn_sklearn.predict(features)
mask_sklearn = pred_sklearn.reshape(h, w).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(mask_sklearn)
plt.title('Máscara KNN - scikit-learn')
plt.axis('off')
plt.show()

## Clasificación con OpenCV (`cv2.ml.KNearest`)

Realiza la misma segmentación, pero utilizando la implementación de K vecinos más cercanos de OpenCV en lugar de la de scikit-learn. El modelo se entrena con los píxeles previamente etiquetados como fondo y primer plano, y luego clasifica todos los píxeles de la imagen según la cercanía de sus características de color a esos ejemplos conocidos. Como resultado, se genera una nueva máscara binaria

In [ ]:
knn_cv = cv2.ml.KNearest_create()
knn_cv.train(X_train.astype(np.float32), cv2.ml.ROW_SAMPLE, y_train.astype(np.float32))

_, results, neighbours, dist = knn_cv.findNearest(features.astype(np.float32), k=K_NEIGHBORS)
mask_cv = results.reshape(h, w).astype(np.uint8)

plt.figure(figsize=(6, 6))
plt.imshow(mask_cv)
plt.title('Máscara KNN - OpenCV')
plt.axis('off')
plt.show()

## Comparación visual


In [ ]:
overlay_sklearn = img_rgb.copy()
overlay_cv = img_rgb.copy()

overlay_sklearn[mask_sklearn == 0] = (0.35 * overlay_sklearn[mask_sklearn == 0]).astype(np.uint8)
overlay_cv[mask_cv == 0] = (0.35 * overlay_cv[mask_cv == 0]).astype(np.uint8)

fig, ax = plt.subplots(2, 3, figsize=(15, 10))

ax[0, 0].imshow(img_rgb)
ax[0, 0].set_title('Imagen RGB original')
ax[0, 0].axis('off')

ax[0, 1].imshow(seed_vis)
ax[0, 1].set_title('Semillas')
ax[0, 1].axis('off')

ax[0, 2].imshow(hsv_to_rgb_vis)
ax[0, 2].set_title('HSV desde RGB filtrada')
ax[0, 2].axis('off')

ax[1, 0].imshow(mask_sklearn)
ax[1, 0].set_title('Máscara scikit-learn')
ax[1, 0].axis('off')

ax[1, 1].imshow(mask_cv)
ax[1, 1].set_title('Máscara OpenCV')
ax[1, 1].axis('off')

ax[1, 2].imshow(np.hstack([overlay_sklearn, overlay_cv]))
ax[1, 2].set_title('Overlay: scikit-learn | OpenCV')
ax[1, 2].axis('off')

plt.tight_layout()
plt.show()
